# Power Outage Model — Training Template

**使用方法**: `cp model_template.ipynb model_你的名字.ipynb`，按 `# TODO` 提示修改。

### 模板自动完成
- ✅ 数据加载 + Phase 1 全套特征工程 (GMM 分层 / 去冗余 / 193 维特征)
- ✅ 训练/验证集划分 + StandardScaler + PyTorch DataLoader
- ✅ wandb 集成 (超参 / 训练曲线 / 评估结果)
- ✅ 统一评估 (县均 RMSE, 24h/48h 窗口) + Phase 1 排行榜自动对比

### 你需要做
- 🔧 §0: 填写 `MODEL_NAME`、选择模型类型、调超参
- 🔧 §3: 如果内置模型不满足需求，修改或替换模型架构
- 🔧 §4: 选择或自定义损失函数

### 内置模型架构 (§3 中选择)
| 模型 | 适用场景 | Config 中选择 |
|------|---------|---------------|
| `DeepLSTM` | 通用时序 baseline，最简单上手 | `MODEL_TYPE = "lstm"` |
| `Seq2SeqAttention` | Encoder-Decoder + Attention | `MODEL_TYPE = "seq2seq"` |
| `TransformerForecaster` | 长程依赖，self-attention | `MODEL_TYPE = "transformer"` |

### 进阶模块 (可叠加，§3.2 中启用)
| 模块 | 做什么 | 启用方式 |
|------|--------|----------|
| GNN 空间模块 | 83 县邻接图，学习停电空间传播 | `GNN_ENABLED = True` |
| Two-Stage | 先分类(有无停电)再回归(停电量) | `TWO_STAGE_ENABLED = True` |
| Ensemble | 加权平均/Stacking 融合多模型 | `ENSEMBLE_ENABLED = True` |

---
## §0 Config

In [ ]:
# ============================================================
# §0 Config — 所有超参数集中在这里
# ============================================================

# ---- TODO: 实验元信息 ----
MODEL_NAME = "MyModel"              # TODO: 填你的模型名 (如 "DeepLSTM_v2", "Transformer_causal")
EXPERIMENT_TAG = "baseline"          # TODO: 本次实验标签 (如 "huber_loss", "3layers", "gnn")

# ---- TODO: 选择基础模型 ----
MODEL_TYPE = "lstm"                  # TODO: "lstm", "seq2seq", "transformer"

# ---- TODO: 进阶模块 (可叠加) ----
GNN_ENABLED = False                  # TODO: True → 接入 GNN 空间模块
TWO_STAGE_ENABLED = False            # TODO: True → 先分类再回归
ENSEMBLE_ENABLED = False             # TODO: True → 融合多模型 (跳过训练，直接到 §7)

# ---- 数据 ----
DATA_DIR = "data"
RESULTS_DIR = "results"
RANDOM_SEED = 42
VALIDATION_SPLIT = 0.2              # 时间序列 80/20
OUTAGE_LAGS = (1, 3, 6, 12, 24)
ROLLING_WINDOWS = (6, 12, 24)
WEATHER_LAGS = (1, 3, 6)

# ---- TODO: 序列参数 ----
SEQ_LEN = 24                        # 输入序列长度 (尝试 24, 48, 72)
HORIZON = 48                        # 预测长度 (24 或 48)

# ---- TODO: 训练参数 ----
BATCH_SIZE = 64                     # 尝试 32, 64, 128, 256
EPOCHS = 50                         # 建议 50-200，配合 early stopping
LEARNING_RATE = 1e-3                # 尝试 1e-4 ~ 1e-2 (Transformer 建议 5e-4)
WEIGHT_DECAY = 1e-5
LR_SCHEDULER = "cosine"             # "cosine", "step", "plateau", None
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0                 # None = 不裁剪

# ---- TODO: 模型架构 ----
HIDDEN_DIM = 128                    # 尝试 64, 128, 256
NUM_LAYERS = 2                      # 尝试 1, 2, 3
DROPOUT = 0.2                       # Transformer 建议 0.1

# LSTM 专用
USE_GRU = False                     # True = GRU 替代 LSTM
BIDIRECTIONAL = False

# Seq2Seq 专用
AUTOREGRESSIVE = False              # True = 逐步解码
TEACHER_FORCING = 0.5              # autoregressive 的 teacher forcing 比例

# Transformer 专用
N_HEADS = 8                         # 需整除 HIDDEN_DIM
DIM_FEEDFORWARD = 512
USE_CAUSAL_MASK = True

# GNN 专用
GNN_HIDDEN = 64
GNN_K = 5                           # k-nearest neighbors
SPATIAL_FUSION = "concat"            # "concat", "add", "gate"

# Two-Stage 专用
CLS_THRESHOLD = 0.5                 # 分类阈值
CLS_POS_WEIGHT = 3.0                # 正样本权重 (因 70% 零值)

# Ensemble 专用
ENSEMBLE_METHOD = "stacking"         # "weighted_avg", "stacking", "per_county_best"

# ---- TODO: 损失函数 ----
LOSS_FN = "mse"                     # "mse", "huber", "weighted_mse"
HUBER_DELTA = 100.0
EXTREME_WEIGHT = 5.0
EXTREME_THRESHOLD = 500

# ---- 设备 (自动检测) ----
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"=== {MODEL_NAME} ({EXPERIMENT_TAG}) ===")
print(f"Model type: {MODEL_TYPE} | Device: {DEVICE}")
print(f"GNN: {GNN_ENABLED} | TwoStage: {TWO_STAGE_ENABLED} | Ensemble: {ENSEMBLE_ENABLED}")
print(f"Horizon: {HORIZON}h | SEQ_LEN: {SEQ_LEN} | LR: {LEARNING_RATE} | Batch: {BATCH_SIZE}")

---
## §1 数据加载 + 特征工程 (自动，不用改)

In [ ]:
# §1.1 导入
import os, sys, time, warnings, platform
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(RANDOM_SEED)

if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS']
elif platform.system() == 'Linux':
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False
print(f"PyTorch {torch.__version__} | {DEVICE}")

In [ ]:
# §1.2 加载数据
ds = xr.open_dataset(os.path.join(DATA_DIR, 'train.nc'))
timestamps = pd.to_datetime(ds.timestamp.values)
locations = list(ds.location.values)
weather_features = list(ds.feature.values)
outage  = ds.out.transpose('timestamp','location').values.astype(float)
tracked = ds.tracked.transpose('timestamp','location').values.astype(float)
weather = ds.weather.transpose('timestamp','location','feature').values.astype(float)
T, L = outage.shape
print(f"T={T}h | L={L} counties | F={len(weather_features)} weather | {timestamps[0].date()} ~ {timestamps[-1].date()}")

In [ ]:
# §1.3 Phase 1 关键产出 (GMM 分层 + 天气特征筛选)
from sklearn.mixture import GaussianMixture
from scipy.stats import spearmanr

# GMM regime thresholds
nonzero_out = outage[outage > 0].flatten()
log_nz = np.log1p(nonzero_out).reshape(-1, 1)
best_k, best_bic = 2, np.inf
for k in range(2, 6):
    gm = GaussianMixture(n_components=k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
    bic = gm.bic(log_nz)
    if bic < best_bic: best_k, best_bic = k, bic
gm_final = GaussianMixture(n_components=best_k, random_state=RANDOM_SEED, max_iter=300).fit(log_nz)
sorted_means = np.sort(gm_final.means_.flatten())
gmm_thresholds = [int(np.expm1((sorted_means[i]+sorted_means[i+1])/2)) for i in range(len(sorted_means)-1)]
print(f"GMM k={best_k}, thresholds={gmm_thresholds}")

# Weather feature selection (Spearman + collinearity dedup)
flat_out = outage.flatten()
sample_n = min(200_000, len(flat_out))
sidx = np.random.choice(len(flat_out), sample_n, replace=False)
corr_dict = {}
for fi, fn in enumerate(weather_features):
    rho, _ = spearmanr(flat_out[sidx], weather[:,:,fi].flatten()[sidx])
    corr_dict[fn] = abs(rho)
corr_series = pd.Series(corr_dict).sort_values(ascending=False)

selected_features = []
for feat in corr_series.index:
    fi = weather_features.index(feat)
    if not any(abs(np.corrcoef(
        weather[:,:,fi].flatten()[sidx],
        weather[:,:,weather_features.index(s)].flatten()[sidx])[0,1]) > 0.85
        for s in selected_features):
        selected_features.append(feat)
top_weather_features = list(corr_series.head(6).index)
print(f"去冗余后: {len(selected_features)} features | Top-6: {top_weather_features}")

In [ ]:
# §1.4 build_features()
def build_features(ds, sel_weather, top_weather, thresholds,
                   w_lags=(1,3,6), o_lags=(1,3,6,12,24), r_wins=(6,12,24)):
    ts = pd.to_datetime(ds.timestamp.values)
    locs = list(ds.location.values)
    fnames = list(ds.feature.values)
    out = ds.out.transpose('timestamp','location').values.astype(float)
    trk = ds.tracked.transpose('timestamp','location').values.astype(float)
    w   = ds.weather.transpose('timestamp','location','feature').values.astype(float)
    Tl, Ll = out.shape
    sel_weather = sel_weather or fnames
    top_weather = top_weather or fnames[:6]
    top_in = [f for f in top_weather if f in fnames]
    thresholds = thresholds or []
    
    rows = []
    for t in range(Tl):
        dt = pd.Timestamp(ts[t])
        h, dow, mo = dt.hour, dt.dayofweek, dt.month
        for li in range(Ll):
            r = {'timestamp': dt, 'location': locs[li], 'out': out[t,li], 'tracked': trk[t,li],
                 'hour': h, 'day_of_week': dow, 'month': mo,
                 'hour_sin': np.sin(2*np.pi*h/24), 'hour_cos': np.cos(2*np.pi*h/24),
                 'dow_sin': np.sin(2*np.pi*dow/7), 'dow_cos': np.cos(2*np.pi*dow/7),
                 'month_sin': np.sin(2*np.pi*mo/12), 'month_cos': np.cos(2*np.pi*mo/12)}
            for lag in o_lags:
                r[f'out_lag_{lag}h'] = out[t-lag,li] if t>=lag else np.nan
                r[f'rate_lag_{lag}h'] = (out[t-lag,li]/max(trk[t-lag,li],1)) if t>=lag else np.nan
            for win in r_wins:
                if t>=win:
                    sl = out[t-win:t,li]
                    r[f'out_rmean_{win}h'],r[f'out_rmax_{win}h'],r[f'out_rstd_{win}h']=sl.mean(),sl.max(),sl.std()
                else:
                    r[f'out_rmean_{win}h']=r[f'out_rmax_{win}h']=r[f'out_rstd_{win}h']=np.nan
            for bw in [6,12,24]:
                r[f'had_outage_{bw}h']=float(np.any(out[max(0,t-bw):t,li]>0)) if t>0 else 0.
            if thresholds:
                prev=out[t-1,li] if t>0 else 0
                r['out_regime']=sum(1 for th in thresholds if prev>th)
                r['out_regime_max_24h']=sum(1 for th in thresholds if (out[t-24:t,li].max() if t>=24 else 0)>th)
            for fn in sel_weather:
                if fn in fnames: r[f'w_{fn}']=w[t,li,fnames.index(fn)]
            for fn in top_in:
                fi=fnames.index(fn)
                for lag in w_lags: r[f'w_{fn}_lag{lag}h']=w[t-lag,li,fi] if t>=lag else np.nan
                for win in r_wins:
                    if t>=win:
                        sw=w[t-win:t,li,fi]; r[f'w_{fn}_rmean_{win}h']=sw.mean(); r[f'w_{fn}_rmax_{win}h']=sw.max()
                    else:
                        r[f'w_{fn}_rmean_{win}h']=r[f'w_{fn}_rmax_{win}h']=np.nan
                r[f'w_{fn}_delta6h']=(w[t,li,fi]-w[t-6,li,fi]) if t>=6 else np.nan
            for a in range(len(top_in)):
                for b in range(a+1,min(len(top_in),4)):
                    r[f'w_{top_in[a]}_x_{top_in[b]}']=w[t,li,fnames.index(top_in[a])]*w[t,li,fnames.index(top_in[b])]
            zs=[]
            for fn in top_in[:6]:
                fi=fnames.index(fn)
                if t>=24:
                    mu,sig=w[t-24:t,li,fi].mean(),w[t-24:t,li,fi].std()+1e-8
                    zs.append(abs((w[t,li,fi]-mu)/sig))
            r['n_weather_anomaly']=sum(1 for z in zs if z>2) if zs else 0
            r['weather_anomaly_score']=np.mean(zs) if zs else 0.
            r['anomaly_score_max_6h']=r['weather_anomaly_score']
            rows.append(r)
    df=pd.DataFrame(rows)
    fg={'target':['out','tracked'],
        'time':[c for c in df.columns if c in ['hour','day_of_week','month','hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos']],
        'outage_lag':[c for c in df.columns if 'out_lag' in c or 'rate_lag' in c],
        'outage_rolling':[c for c in df.columns if (c.startswith('out_r') or c.startswith('had_'))],
        'outage_regime':[c for c in df.columns if 'regime' in c],
        'weather_raw':[c for c in df.columns if c.startswith('w_') and not any(x in c for x in ['lag','rmean','rmax','delta','_x_'])],
        'weather_lag':[c for c in df.columns if c.startswith('w_') and 'lag' in c],
        'weather_rolling':[c for c in df.columns if c.startswith('w_') and any(x in c for x in ['rmean','rmax','delta'])],
        'weather_interaction':[c for c in df.columns if '_x_' in c],
        'storm_indicator':['n_weather_anomaly','weather_anomaly_score','anomaly_score_max_6h']}
    return df, fg

print('构造特征中 (~2min)...')
t0=time.time()
df, feat_groups = build_features(ds, selected_features, top_weather_features, gmm_thresholds,
                                  WEATHER_LAGS, OUTAGE_LAGS, ROLLING_WINDOWS)
feature_cols=[]
for g,ns in feat_groups.items():
    if g!='target': feature_cols.extend([c for c in ns if c in df.columns])
feature_cols=list(dict.fromkeys(feature_cols))
print(f'✓ {time.time()-t0:.0f}s | {df.shape[0]:,} rows × {len(feature_cols)} features')

In [ ]:
# §1.5 训练/验证划分 + 标准化
split_idx = int(T * (1 - VALIDATION_SPLIT))
split_time = timestamps[split_idx]
train_df = df[df['timestamp'] < split_time].dropna(subset=feature_cols).copy()
val_df = df[df['timestamp'] >= split_time].copy()
for c in feature_cols: val_df[c] = val_df[c].fillna(0)

scaler = StandardScaler().fit(train_df[feature_cols].values)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
print(f'y_train: mean={train_df["out"].mean():.1f}, nonzero={train_df["out"].gt(0).mean():.1%}')
print(f'y_val:   mean={val_df["out"].mean():.1f}, nonzero={val_df["out"].gt(0).mean():.1%}')

---
## §2 DataLoader

In [ ]:
# §2.1 DataFrame → 3D 张量
def df_to_3d(sub_df, locs, feat_cols, scaler_obj):
    ts_list = sorted(sub_df['timestamp'].unique())
    ts2i = {ts:i for i,ts in enumerate(ts_list)}
    loc2i = {l:i for i,l in enumerate(locs)}
    Ts, Ls, Fs = len(ts_list), len(locs), len(feat_cols)
    X = np.zeros((Ts,Ls,Fs), dtype=np.float32)
    Y = np.zeros((Ts,Ls), dtype=np.float32)
    sorted_df = sub_df.sort_values(['timestamp','location'])
    scaled = scaler_obj.transform(sorted_df[feat_cols].fillna(0).values)
    for idx, (_, row) in enumerate(sorted_df.iterrows()):
        ti, li = ts2i.get(row['timestamp']), loc2i.get(row['location'])
        if ti is not None and li is not None:
            X[ti,li,:] = scaled[idx]
            Y[ti,li] = row['out']
    return X, Y

X_train_3d, Y_train_3d = df_to_3d(train_df, locations, feature_cols, scaler)
X_val_3d, Y_val_3d = df_to_3d(val_df, locations, feature_cols, scaler)
print(f'X_train: {X_train_3d.shape} | X_val: {X_val_3d.shape}')

In [ ]:
# §2.2 PyTorch Dataset (支持 per_county 和 all_counties 两种模式)
class OutageDataset(Dataset):
    """
    mode='per_county':    每样本 = 单县时间窗 → (seq_len, F) → (horizon,)
    mode='all_counties':  每样本 = 全部县同一时间窗 → (seq_len, L, F) → (horizon, L)
    """
    def __init__(self, X, Y, seq_len, horizon, mode='per_county'):
        self.X, self.Y = X, Y
        self.seq_len, self.horizon, self.mode = seq_len, horizon, mode
        self.Td, self.Ld, self.Fd = X.shape
        self.starts = list(range(self.Td - seq_len - horizon + 1))
        self.n = len(self.starts) * (self.Ld if mode == 'per_county' else 1)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        if self.mode == 'per_county':
            ti, li = idx // self.Ld, idx % self.Ld
            t = self.starts[ti]
            x = self.X[t:t+self.seq_len, li, :]                     # (S, F)
            y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon, li]  # (H,)
        else:
            t = self.starts[idx]
            x = self.X[t:t+self.seq_len]                            # (S, L, F)
            y = self.Y[t+self.seq_len:t+self.seq_len+self.horizon]  # (H, L)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# GNN 需要 all_counties，其余用 per_county
DS_MODE = 'all_counties' if GNN_ENABLED else 'per_county'

train_ds = OutageDataset(X_train_3d, Y_train_3d, SEQ_LEN, HORIZON, mode=DS_MODE)
val_ds   = OutageDataset(X_val_3d,   Y_val_3d,   SEQ_LEN, HORIZON, mode=DS_MODE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f'Mode: {DS_MODE} | Train: {len(train_ds):,} | Val: {len(val_ds):,}')
print(f'Batch X: {xb.shape} | Batch Y: {yb.shape}')

---
## §3 模型定义

三个内置架构 + 三个进阶模块，通过 §0 的 Config 自动选择。

也可以完全不用内置的，删掉 §3 自己写。

In [ ]:
# §3.1 三个基础模型

class DeepLSTM(nn.Module):
    """多层 LSTM/GRU + LayerNorm + 可选 Bidirectional"""
    def __init__(self, input_dim, hidden_dim, num_layers, horizon,
                 dropout=0.2, use_gru=False, bidirectional=False):
        super().__init__()
        ndir = 2 if bidirectional else 1
        RNN = nn.GRU if use_gru else nn.LSTM
        self.rnn = RNN(input_dim, hidden_dim, num_layers, batch_first=True,
                       dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        self.norm = nn.LayerNorm(hidden_dim * ndir)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * ndir, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon))

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(self.norm(out[:, -1, :]))


class Seq2SeqAttention(nn.Module):
    """Encoder LSTM + Bahdanau Attention + Decoder head"""
    def __init__(self, input_dim, hidden_dim, num_layers, horizon, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.attn_We = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_Wd = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attn_v  = nn.Linear(hidden_dim, 1, bias=False)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon))

    def attention(self, enc_out, h):
        e = torch.tanh(self.attn_We(enc_out) + self.attn_Wd(h).unsqueeze(1))
        w = torch.softmax(self.attn_v(e).squeeze(-1), dim=1)
        ctx = torch.bmm(w.unsqueeze(1), enc_out).squeeze(1)
        return ctx, w

    def forward(self, x, return_attention=False):
        enc_out, (h_n, _) = self.encoder(x)
        ctx, attn_w = self.attention(enc_out, h_n[-1])
        out = self.head(torch.cat([h_n[-1], ctx], dim=1))
        return (out, attn_w) if return_attention else out


class TransformerForecaster(nn.Module):
    """Transformer Encoder + sinusoidal PE + linear head"""
    def __init__(self, input_dim, d_model, n_heads, num_layers, horizon,
                 dim_ff=512, dropout=0.1, max_len=200, causal=True):
        super().__init__()
        self.causal = causal
        self.proj = nn.Linear(input_dim, d_model)
        # positional encoding
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2]) if d_model % 2 else torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.drop = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_ff, dropout,
                                               batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, horizon))

    def forward(self, x):
        B, T, _ = x.shape
        x = self.drop(self.proj(x) + self.pe[:, :T])
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool() if self.causal else None
        x = self.encoder(x, mask=mask)
        return self.head(x[:, -1, :])


print('✓ DeepLSTM / Seq2SeqAttention / TransformerForecaster 已定义')

In [ ]:
# §3.2 进阶模块

# ======================== GNN 空间模块 ========================
class GCNLayer(nn.Module):
    """手写 GCN (不依赖 torch-geometric)"""
    def __init__(self, in_d, out_d):
        super().__init__()
        self.W = nn.Linear(in_d, out_d, bias=False)
        self.b = nn.Parameter(torch.zeros(out_d))
    def forward(self, x, A):
        return torch.relu(torch.matmul(A, self.W(x)) + self.b)

class SpatialWrapper(nn.Module):
    """
    给任意时序模型叠加 GNN 空间模块。
    输入 (B, T, L, F) → 每县独立过 backbone → GCN 传播 → 融合 → (B, L, horizon)
    """
    def __init__(self, backbone, hidden_dim, horizon, n_counties,
                 gnn_hidden=64, fusion='concat', dropout=0.2):
        super().__init__()
        self.backbone = backbone  # 任何 (B, T, F) → (B, H) 的模型
        self.gcn1 = GCNLayer(hidden_dim, gnn_hidden)
        self.gcn2 = GCNLayer(gnn_hidden, gnn_hidden)
        self.fusion = fusion
        head_in = (hidden_dim + gnn_hidden) if fusion == 'concat' else hidden_dim
        if fusion == 'gate':
            self.gate = nn.Linear(hidden_dim + gnn_hidden, hidden_dim)
        self.head = nn.Sequential(
            nn.LayerNorm(head_in), nn.Linear(head_in, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, horizon))
    
    def forward(self, x, adj):
        B, T, L_loc, F = x.shape
        # 每县独立编码
        x_flat = x.permute(0,2,1,3).reshape(B*L_loc, T, F)
        # 只取 backbone 的 encoder hidden (用 hook 或直接拿 rnn output)
        rnn_out, _ = self.backbone.rnn(x_flat) if hasattr(self.backbone, 'rnn') else self.backbone.encoder(x_flat)
        h_lstm = rnn_out[:, -1, :].reshape(B, L_loc, -1)  # (B, L, H)
        h_gnn = self.gcn2(self.gcn1(h_lstm, adj), adj)     # (B, L, gnn_h)
        if self.fusion == 'concat':
            h = torch.cat([h_lstm, h_gnn], dim=-1)
        elif self.fusion == 'gate':
            g = torch.sigmoid(self.gate(torch.cat([h_lstm, h_gnn], dim=-1)))
            h = g * h_lstm + (1-g) * h_gnn
        else:
            h = h_lstm + h_gnn
        return self.head(h)  # (B, L, horizon)

def build_adj(locations, k=5):
    """TODO: 用真实 FIPS→经纬度 构建邻接矩阵。当前占位: 均匀全连接。"""
    n = len(locations)
    return (torch.ones(n, n) / n)


# ======================== Two-Stage ========================
class TwoStageWrapper:
    """
    Stage 1: 分类 — 该 (县, 时间步) 是否停电
    Stage 2: 回归 — 仅对 "会停电" 的位置预测停电量
    推理: pred = (prob > threshold) * regression_pred
    """
    def __init__(self, cls_model, reg_model, threshold=0.5):
        self.cls = cls_model
        self.reg = reg_model
        self.threshold = threshold
    
    def predict(self, x):
        with torch.no_grad():
            prob = torch.sigmoid(self.cls(x))
            mask = (prob > self.threshold).float()
            reg_pred = torch.clamp(self.reg(x), min=0)
            return mask * reg_pred


# ======================== Ensemble ========================
class EnsemblePredictor:
    """
    融合多模型验证集预测。
    method='weighted_avg':   权重 ∝ 1/RMSE
    method='stacking':       每县独立 RidgeCV
    method='per_county_best': 每县选 RMSE 最低的模型
    """
    def __init__(self, method='stacking'):
        self.method = method
        self.preds, self.rmses, self.county_rmses = {}, {}, {}
    
    def add(self, name, val_pred, val_true, locs):
        self.preds[name] = val_pred
        cr = {locs[i]: np.sqrt(np.mean((val_true[:,i]-val_pred[:,i])**2)) for i in range(len(locs))}
        self.county_rmses[name] = cr
        self.rmses[name] = np.mean(list(cr.values()))
        print(f'  + {name}: RMSE={self.rmses[name]:.4f}')
    
    def fit(self, val_true, locs):
        names = list(self.preds.keys())
        if self.method == 'weighted_avg':
            inv = np.array([1./self.rmses[n] for n in names])
            self.weights = dict(zip(names, inv/inv.sum()))
            print(f'  Weights: {self.weights}')
        elif self.method == 'stacking':
            from sklearn.linear_model import RidgeCV
            self.ridge = {}
            for i, loc in enumerate(locs):
                X = np.column_stack([self.preds[n][:,i] for n in names])
                self.ridge[loc] = RidgeCV(alphas=[.01,.1,1,10]).fit(X, val_true[:,i])
            print(f'  Stacking fitted for {len(locs)} counties')
        elif self.method == 'per_county_best':
            self.best = {loc: min(names, key=lambda n: self.county_rmses[n][loc]) for loc in locs}
            from collections import Counter
            print(f'  Best per county: {dict(Counter(self.best.values()))}')
    
    def predict(self, test_preds, locs):
        names = list(self.preds.keys())
        sample = list(test_preds.values())[0]
        res = np.zeros_like(sample)
        if self.method == 'weighted_avg':
            for n in names: res += self.weights[n] * test_preds[n]
        elif self.method == 'stacking':
            for i, loc in enumerate(locs):
                X = np.column_stack([test_preds[n][:,i] for n in names])
                res[:,i] = self.ridge[loc].predict(X)
        elif self.method == 'per_county_best':
            for i, loc in enumerate(locs):
                res[:,i] = test_preds[self.best[loc]][:,i]
        return np.clip(res, 0, None)


print('✓ GCN/SpatialWrapper / TwoStageWrapper / EnsemblePredictor 已定义')

In [ ]:
# §3.3 自动实例化
INPUT_DIM = len(feature_cols)

# ---- 构建 backbone ----
if MODEL_TYPE == 'lstm':
    backbone = DeepLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON,
                        DROPOUT, USE_GRU, BIDIRECTIONAL)
elif MODEL_TYPE == 'seq2seq':
    backbone = Seq2SeqAttention(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, HORIZON, DROPOUT)
elif MODEL_TYPE == 'transformer':
    backbone = TransformerForecaster(INPUT_DIM, HIDDEN_DIM, N_HEADS, NUM_LAYERS, HORIZON,
                                     DIM_FEEDFORWARD, DROPOUT, causal=USE_CAUSAL_MASK)
else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

# ---- 叠加 GNN? ----
if GNN_ENABLED:
    adj_matrix = build_adj(locations, GNN_K).to(DEVICE)
    model = SpatialWrapper(backbone, HIDDEN_DIM, HORIZON, L,
                           GNN_HIDDEN, SPATIAL_FUSION, DROPOUT).to(DEVICE)
    print(f'Model: {MODEL_TYPE} + GNN ({SPATIAL_FUSION})')
else:
    model = backbone.to(DEVICE)
    adj_matrix = None
    print(f'Model: {MODEL_TYPE}')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Params: {n_params:,} | Device: {DEVICE}')
print(model)

---
## §4 损失函数

In [ ]:
# §4 损失函数
class WeightedMSE(nn.Module):
    def __init__(self, threshold=500, weight=5.0):
        super().__init__()
        self.threshold, self.weight = threshold, weight
    def forward(self, pred, target):
        w = torch.where(target > self.threshold, self.weight, 1.0)
        return (w * (pred - target)**2).mean()

if LOSS_FN == 'huber':
    criterion = nn.HuberLoss(delta=HUBER_DELTA)
elif LOSS_FN == 'weighted_mse':
    criterion = WeightedMSE(EXTREME_THRESHOLD, EXTREME_WEIGHT)
else:
    criterion = nn.MSELoss()

# Two-Stage 的分类 loss
cls_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([CLS_POS_WEIGHT]).to(DEVICE)) if TWO_STAGE_ENABLED else None

print(f'Loss: {criterion}')
if cls_criterion: print(f'CLS Loss: {cls_criterion}')

---
## §5 wandb

In [ ]:
# §5 wandb 初始化
WANDB_ENABLED = False
try:
    import wandb
    from dotenv import load_dotenv; load_dotenv()
    api_key = os.getenv('WANDB_API_KEY')
    entity  = os.getenv('WANDB_ENTITY')
    user    = os.getenv('WANDB_USERNAME', 'unknown')
    if api_key and entity and entity != 'your_team_name':
        wandb.login(key=api_key)
        run_name = f'{user}_{MODEL_NAME}_{EXPERIMENT_TAG}_{datetime.now().strftime("%m%d_%H%M")}'
        wandb.init(project='MLPS-Power-Outage', entity=entity, name=run_name,
                   tags=['phase2', MODEL_TYPE, EXPERIMENT_TAG],
                   config={k: v for k, v in globals().items()
                           if isinstance(v, (int, float, str, bool, list, tuple)) and k.isupper()})
        WANDB_ENABLED = True
        print(f'✓ wandb: {run_name}')
    else:
        print('⚠ wandb 未配置')
except Exception as e:
    print(f'⚠ wandb 不可用: {e}')

---
## §6 训练循环

如果 `ENSEMBLE_ENABLED=True`，跳过此节直接到 §7。

In [ ]:
# §6.1 优化器 + 调度器
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
if LR_SCHEDULER == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=LEARNING_RATE*0.01)
elif LR_SCHEDULER == 'step':
    scheduler = optim.lr_scheduler.StepLR(optimizer, 20, 0.5)
elif LR_SCHEDULER == 'plateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', 0.5, patience=5)
else:
    scheduler = None
print(f'Optimizer: AdamW | Scheduler: {LR_SCHEDULER}')

In [ ]:
# §6.2 训练 + 验证
if ENSEMBLE_ENABLED:
    print('ENSEMBLE_ENABLED=True, 跳过训练 → 直接到 §7')
else:
    best_val_rmse, best_epoch, patience_ctr = float('inf'), 0, 0
    history = {'train_loss':[], 'val_loss':[], 'val_rmse':[], 'lr':[]}
    t_start = time.time()

    print(f'\n{"="*60}')
    print(f'Training: {MODEL_NAME} | {EPOCHS} epochs | {DEVICE}')
    print(f'{"="*60}\n')

    for epoch in range(1, EPOCHS + 1):
        # ---- Train ----
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            # GNN 模型需要传 adj_matrix
            pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
            # Two-Stage 分类阶段: target 是 (y>0).float()
            if TWO_STAGE_ENABLED and cls_criterion is not None:
                loss = cls_criterion(pred, (yb > 0).float())
            else:
                loss = criterion(pred, yb)
            loss.backward()
            if GRADIENT_CLIP: nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            losses.append(loss.item())
        train_loss = np.mean(losses)

        # ---- Validate ----
        model.eval()
        vl, vp, vt = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb, adj_matrix) if GNN_ENABLED else model(xb)
                if TWO_STAGE_ENABLED and cls_criterion is not None:
                    vl.append(cls_criterion(pred, (yb>0).float()).item())
                else:
                    vl.append(criterion(pred, yb).item())
                vp.append(pred.cpu().numpy()); vt.append(yb.cpu().numpy())
        val_loss = np.mean(vl)
        all_p, all_t = np.concatenate(vp), np.concatenate(vt)
        val_rmse = np.sqrt(np.mean((all_p - all_t)**2))

        # ---- LR schedule ----
        cur_lr = optimizer.param_groups[0]['lr']
        if scheduler:
            scheduler.step(val_rmse) if LR_SCHEDULER == 'plateau' else scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)
        history['lr'].append(cur_lr)

        if WANDB_ENABLED:
            wandb.log({'epoch': epoch, 'train/loss': train_loss,
                       'val/loss': val_loss, 'val/rmse': val_rmse, 'lr': cur_lr})

        if val_rmse < best_val_rmse:
            best_val_rmse, best_epoch, patience_ctr = val_rmse, epoch, 0
            os.makedirs(RESULTS_DIR, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'))
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or epoch == 1 or patience_ctr == 0:
            m = '★' if patience_ctr == 0 else ' '
            print(f'  E{epoch:3d}/{EPOCHS} | TrLoss:{train_loss:.4f} ValRMSE:{val_rmse:.4f} '
                  f'Best:{best_val_rmse:.4f}@{best_epoch} LR:{cur_lr:.6f} {m}')

        if patience_ctr >= EARLY_STOPPING_PATIENCE:
            print(f'\n  ⏹ Early stop @ epoch {epoch}')
            break

    total_time = time.time() - t_start
    print(f'\n✓ {total_time:.0f}s | Best: {best_val_rmse:.4f} @ epoch {best_epoch}')

In [ ]:
# §6.3 训练曲线
if not ENSEMBLE_ENABLED:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].axvline(best_epoch-1, c='r', ls='--', alpha=.5); axes[0].legend()
    axes[0].set(xlabel='Epoch', ylabel='Loss', yscale='log', title='Loss')
    axes[1].plot(history['val_rmse'], c='#e74c3c')
    axes[1].axhline(best_val_rmse, c='g', ls='--', alpha=.5, label=f'Best={best_val_rmse:.2f}')
    axes[1].legend(); axes[1].set(xlabel='Epoch', ylabel='RMSE', title='Val RMSE')
    axes[2].plot(history['lr'], c='#3498db')
    axes[2].set(xlabel='Epoch', ylabel='LR', yscale='log', title='Learning Rate')
    plt.suptitle(f'{MODEL_NAME} ({EXPERIMENT_TAG})', fontweight='bold'); plt.tight_layout(); plt.show()
    if WANDB_ENABLED: wandb.log({'charts/curves': wandb.Image(fig)})

---
## §7 统一评估 (县均 RMSE)

In [ ]:
# §7.1 评估函数
def evaluate(y_true, y_pred, locs, name='Model'):
    cr = {locs[i]: np.sqrt(np.mean((y_true[:,i]-y_pred[:,i])**2)) for i in range(len(locs))}
    avg = np.mean(list(cr.values()))
    print(f'=== {name}: RMSE={avg:.4f} ===')
    for loc, r in sorted(cr.items(), key=lambda x: -x[1])[:5]:
        print(f'  {loc}: {r:.4f}')
    return avg, cr

def evaluate_horizons(y_true, y_pred, locs, name):
    res = {}
    if y_true.shape[0] >= 24:
        r24, _ = evaluate(y_true[:24], y_pred[:24], locs, f'{name}_24h')
        res[f'{name}_24h'] = r24
    if y_true.shape[0] >= 48:
        r48, _ = evaluate(y_true[:48], y_pred[:48], locs, f'{name}_48h')
        res[f'{name}_48h'] = r48
    rf, cr = evaluate(y_true, y_pred, locs, f'{name}_full')
    res[f'{name}_full'] = rf
    return res, cr

print('✓ evaluate / evaluate_horizons')

In [ ]:
# §7.2 生成验证集预测
if not ENSEMBLE_ENABLED:
    model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_best.pt'),
                                     map_location=DEVICE, weights_only=True))
    model.eval()

    Y_pred_val = np.zeros_like(Y_val_3d)
    counts = np.zeros_like(Y_val_3d)

    with torch.no_grad():
        X_full = np.concatenate([X_train_3d[-SEQ_LEN:], X_val_3d], axis=0)
        
        if GNN_ENABLED:
            # all_counties: 一次预测所有县
            for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                xin = torch.tensor(X_full[t:t+SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                pred = model(xin, adj_matrix).cpu().numpy().squeeze(0)  # (horizon, L)
                vs = t + SEQ_LEN - SEQ_LEN
                ve = min(vs + HORIZON, Y_val_3d.shape[0])
                nf = ve - max(vs, 0)
                if vs >= 0 and nf > 0:
                    Y_pred_val[vs:ve] += pred[:nf]
                    counts[vs:ve] += 1
        else:
            # per_county
            for li in range(L):
                for t in range(0, X_full.shape[0]-SEQ_LEN-HORIZON+1, HORIZON):
                    xin = torch.tensor(X_full[t:t+SEQ_LEN, li], dtype=torch.float32).unsqueeze(0).to(DEVICE)
                    pred = model(xin).cpu().numpy().flatten()
                    vs = t + SEQ_LEN - SEQ_LEN
                    ve = min(vs + HORIZON, Y_val_3d.shape[0])
                    nf = ve - max(vs, 0)
                    if vs >= 0 and nf > 0:
                        Y_pred_val[vs:ve, li] += pred[:nf]
                        counts[vs:ve, li] += 1

    counts[counts == 0] = 1
    Y_pred_val = np.clip(Y_pred_val / counts, 0, None)
    
    # 保存预测 (方便 Ensemble 使用)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    np.save(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_val_pred.npy'), Y_pred_val)
    print(f'Val pred: {Y_pred_val.shape} | saved to {RESULTS_DIR}/{MODEL_NAME}_val_pred.npy')

In [ ]:
# §7.3 评估 + 排行榜对比
BASELINES = {
    'SARIMAX_24h': 27.91, 'SARIMAX_48h': 20.13,
    'HistGBM_FE_24h': 54.37, 'HistGBM_FE_48h': 44.02,
    'HistAvg_24h': 93.03, 'HistAvg_48h': 73.55,
    'Seq2Seq_v1_24h': 100.69, 'Seq2Seq_v1_48h': 109.38,
    'Persistence_24h': 129.17, 'Persistence_48h': 117.76,
}

if ENSEMBLE_ENABLED:
    # TODO: 加载各模型的 val_pred.npy
    ensemble = EnsemblePredictor(ENSEMBLE_METHOD)
    # ensemble.add('DeepLSTM', np.load('results/DeepLSTM_val_pred.npy'), Y_val_3d, locations)
    # ensemble.add('Seq2Seq', np.load('results/Seq2Seq_val_pred.npy'), Y_val_3d, locations)
    # ensemble.add('Transformer', np.load('results/Transformer_val_pred.npy'), Y_val_3d, locations)
    # ensemble.fit(Y_val_3d, locations)
    # Y_pred_val = ensemble.predict({n: ensemble.preds[n] for n in ensemble.preds}, locations)
    print('⚠ Ensemble: 请取消上方注释, 填入各模型预测路径')
    model_results = {}
else:
    model_results, model_county_rmses = evaluate_horizons(Y_val_3d, Y_pred_val, locations, MODEL_NAME)

all_results = {**BASELINES, **model_results}
print(f'\n{"="*70}')
print('RMSE 排行榜')
print(f'{"="*70}')
for h in ['24h', '48h']:
    print(f'\n--- {h} ---')
    hr = {k:v for k,v in all_results.items() if h in k}
    for name, rmse in sorted(hr.items(), key=lambda x: x[1]):
        tag = '  ◀ YOU' if MODEL_NAME in name else ''
        print(f'  {name:25s}: {rmse:10.4f}{tag}')

if WANDB_ENABLED:
    for k,v in model_results.items(): wandb.summary[f'eval/{k}'] = v

In [ ]:
# §7.4 排行榜可视化
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, h in zip(axes, ['24h', '48h']):
    hr = sorted([(k,v) for k,v in all_results.items() if h in k], key=lambda x: x[1])
    if not hr: continue
    names, rmses = zip(*hr)
    colors = ['#e74c3c' if MODEL_NAME in n else '#2196F3' if 'SARIMAX' in n
              else '#4CAF50' if 'GBM' in n else '#FF9800' if 'Seq2Seq' in n else '#bdbdbd' for n in names]
    bars = ax.barh(range(len(names)), rmses, color=colors, alpha=.85)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel('县均 RMSE'); ax.set_title(f'{h}', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    for i, (b, v) in enumerate(zip(bars, rmses)):
        ax.text(v + max(rmses)*.01, i, f'{v:.2f}', va='center', fontsize=9)
plt.suptitle(f'排行榜 — {MODEL_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
if WANDB_ENABLED: wandb.log({'charts/leaderboard': wandb.Image(fig)})

---
## §8 提交文件生成

In [ ]:
# §8 提交文件
GENERATE_SUBMISSION = False  # TODO: 确认模型效果后改为 True

if GENERATE_SUBMISSION:
    # TODO: 加载 test data + 推理 + 生成 CSV
    # ds_test_24h = xr.open_dataset(os.path.join(DATA_DIR, 'test_24h_demo.nc'))
    # ds_test_48h = xr.open_dataset(os.path.join(DATA_DIR, 'test_48h_demo.nc'))
    # ...
    print('⚠ 请完成 test set 推理代码')
else:
    print('跳过 (GENERATE_SUBMISSION=False)')

In [ ]:
# §9 总结 + wandb finish
print(f'\n{"="*60}')
print(f'{MODEL_NAME} ({EXPERIMENT_TAG}) 实验总结')
print(f'{"="*60}')
if not ENSEMBLE_ENABLED:
    print(f'  Best Val RMSE: {best_val_rmse:.4f} @ epoch {best_epoch}')
    print(f'  训练时间: {total_time:.0f}s | 参数: {n_params:,}')
for k,v in sorted(model_results.items(), key=lambda x: x[1]):
    print(f'  {k}: {v:.4f}')

if WANDB_ENABLED:
    if not ENSEMBLE_ENABLED:
        wandb.summary['best_val_rmse'] = best_val_rmse
        wandb.summary['best_epoch'] = best_epoch
        wandb.summary['total_time_s'] = total_time
    wandb.finish()
    print('✓ wandb finished')

print(f'\n模型: {RESULTS_DIR}/{MODEL_NAME}_best.pt')
print(f'预测: {RESULTS_DIR}/{MODEL_NAME}_val_pred.npy')

---
## 附录

### 调参优先级
1. **Loss** — `huber` 或 `weighted_mse` (极端值友好)
2. **LR** — {1e-4, 3e-4, 1e-3, 3e-3}，Transformer 偏小
3. **Depth** — layers × hidden_dim
4. **SEQ_LEN** — {24, 48, 72}，Attention/Transformer 受益于长序列
5. **进阶模块** — GNN, Two-Stage, Ensemble

### Phase 1 关键发现
- 停电滞后特征贡献 60%+ → 确保 `OUTAGE_LAGS` 包含 24h
- 极端县 (26125, 26163) RMSE 极高 → 尝试县级 re-weighting
- 70% 零值 → Two-Stage (先分类再回归) 可能有效
- 天气 rolling 特征比 raw 更有用 → `weather_rolling` 组

### 分工交付协议
每人完成后提供:
1. `results/{MODEL_NAME}_best.pt` — checkpoint
2. `results/{MODEL_NAME}_val_pred.npy` — 验证集预测 `(T_val, L)`
3. wandb 上的完整实验记录
4. 24h / 48h RMSE 数字 → 汇总到排行榜